# SAXS Dataset Visualizer

Pick a dataset from the `smartt.data_containers` registry, load whichever of
`main` / `remount` / `combined` DataContainers it has, and browse the raw
projections interactively. Also reports basic statistics (range, mean/std,
percentiles) and histograms of the projection intensities and diode values.

Change `DATASET` in the configuration cell below and re-run from there
downstream to switch datasets.

In [1]:
import sys
sys.path.insert(0, '/myhome/smartt')

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

from smartt.data_containers import get_dataset, REGISTRY

INFO:Setting the number of threads to 8. If your physical cores are fewer than this number, you may want to use numba.set_num_threads(n), and os.environ["OPENBLAS_NUM_THREADS"] = f"{n}" to set the number of threads to the number of physical cores n.
INFO:Setting numba log level to WARNING.


## Configuration

In [11]:
print('Available datasets:', sorted(REGISTRY))

DATASET = 'auditory-ossicle'   # <- change me, then re-run from the next cell down

Available datasets: ['auditory-ossicle', 'b411', 'cf-carolina', 'cf-peek', 'fiber-synthetic', 'fiber-synthetic-full', 'frogbone', 'nielsen-m', 'nielsen-mammoth', 'nielsen-t', 'plastic-plasmonics', 'synthetic-b411', 'zenodo']


## Load dataset

In [12]:
ds = get_dataset(DATASET)
dc_types = ds.available_dc_types()
dcs = {t: ds.get_dc(t) for t in dc_types}

print(f'{DATASET!r} -> dc_types available: {dc_types}\n')
for t, dc in dcs.items():
    n = len(dc.projections)
    print(f'  {t:9s}  n_projections={n:4d}  '
          f'volume_shape={tuple(dc.geometry.volume_shape)}  '
          f'projection_shape={tuple(dc.geometry.projection_shape)}  '
          f'full_circle_covered={dc.geometry.full_circle_covered}')

INFO:Rotation matrices were loaded from the input file.
INFO:No sample geometry information was found. Default mumott geometry assumed.
INFO:No detector geometry information was found. Default mumott geometry assumed.
'auditory-ossicle' -> dc_types available: ['main']

  main       n_projections= 306  volume_shape=(np.int64(83), np.int64(144), np.int64(306))  projection_shape=(np.int64(83), np.int64(144))  full_circle_covered=False


## Projection viewer

Scroll through projections (and detector-angle channels) for any of the
available `dc_type`s. `idx`/`channel` sliders are shared across `dc_type`s and
clamped to whatever range the selected one actually has (smaller datasets
just stop scrolling early — noted in the title when clamped).

In [ ]:
_max_idx = max(len(dc.projections) for dc in dcs.values()) - 1
_max_ch  = max(dc.projections[0].data.shape[-1] for dc in dcs.values()) - 1


def _fmt_deg(rad):
    return f'{np.degrees(rad):.1f}°' if rad is not None else 'n/a'


@interact(
    dc_type=widgets.Dropdown(options=dc_types, description='dc_type'),
    idx=widgets.IntSlider(min=0, max=_max_idx, step=1, continuous_update=False, description='projection'),
    channel=widgets.IntSlider(min=0, max=_max_ch, step=1, continuous_update=False, description='channel'),
)
def _view_projection(dc_type, idx, channel):
    dc = dcs[dc_type]
    n = len(dc.projections)
    i = min(idx, n - 1)
    p = dc.projections[i]
    m = p.data.shape[-1]
    c = min(channel, m - 1)

    inner_deg = _fmt_deg(p.inner_angle)
    outer_deg = _fmt_deg(p.outer_angle)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    vmax = np.percentile(p.data, 99)

    im0 = axes[0].imshow(p.diode.T, origin='lower', cmap='inferno')
    axes[0].set_title('diode')
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(p.data[..., c].T, origin='lower', cmap='viridis', vmin=0, vmax=vmax)
    axes[1].set_title(f'channel {c}/{m - 1}')
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    note = '' if (i == idx and c == channel) else '  (clamped to range)'
    fig.suptitle(f'{DATASET} / {dc_type}   proj={i}/{n - 1}   '
                 f'phi={inner_deg}   alpha={outer_deg}{note}')
    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='dc_type', options=('main',), value='main'), IntSlider(value=0, con…

### All channels at once

Quick overview of every detector-angle channel for a single projection —
handy for spotting angle-dependent artifacts.

In [ ]:
def plot_all_channels(dc, idx, label=''):
    p = dc.projections[idx]
    n_ch = p.data.shape[-1]
    fig, axes = plt.subplots(1, n_ch + 1, figsize=(2.4 * (n_ch + 1), 3))

    im = axes[0].imshow(p.diode.T, origin='lower', cmap='inferno')
    axes[0].set_title('diode')
    plt.colorbar(im, ax=axes[0], fraction=0.046)

    vmax = np.percentile(p.data * p.weights, 100)
    for c, ax in enumerate(axes[1:]):
        im = ax.imshow(p.data[..., c].T * p.weights[..., c].T, origin='lower', cmap='viridis', vmin=0, vmax=vmax)
        ax.set_title(f'ch {c}')
        plt.colorbar(im, ax=ax, fraction=0.046)

    inner_deg = _fmt_deg(p.inner_angle)
    outer_deg = _fmt_deg(p.outer_angle)
    fig.suptitle(f'{label}  proj={idx}  phi={inner_deg}  alpha={outer_deg}', y=1.02)
    plt.tight_layout()
    plt.show()


@interact(
    dc_type=widgets.Dropdown(options=dc_types, description='dc_type'),
    idx=widgets.IntSlider(min=0, max=_max_idx, step=1, continuous_update=False, description='projection'),
)
def _view_all_channels(dc_type, idx):
    dc = dcs[dc_type]
    i = min(idx, len(dc.projections) - 1)
    plot_all_channels(dc, i, label=f'{DATASET} / {dc_type}')

interactive(children=(Dropdown(description='dc_type', options=('main',), value='main'), IntSlider(value=0, con…

## Statistics

In [16]:
def projection_stats(dc, sample_stride=1):
    data = dc.data[::sample_stride]
    return {
        'shape':         data.shape,
        'dtype':         data.dtype,
        'min':           float(data.min()),
        'max':           float(data.max()),
        'mean':          float(data.mean()),
        'std':           float(data.std()),
        'p01':           float(np.percentile(data, 1)),
        'p50':           float(np.percentile(data, 50)),
        'p99':           float(np.percentile(data, 99)),
        'frac_zero':     float(np.mean(data == 0)),
        'frac_negative': float(np.mean(data < 0)),
    }


print(f'{"dc_type":9s} {"shape":20s} {"dtype":8s} {"min":>10s} {"max":>10s} '
      f'{"mean":>10s} {"std":>10s} {"p01":>10s} {"p99":>10s} {"%zero":>7s}')
for t, dc in dcs.items():
    s = projection_stats(dc)
    print(f'{t:9s} {str(s["shape"]):20s} {str(s["dtype"]):8s} '
          f'{s["min"]:10.3g} {s["max"]:10.3g} {s["mean"]:10.3g} {s["std"]:10.3g} '
          f'{s["p01"]:10.3g} {s["p99"]:10.3g} {100 * s["frac_zero"]:6.2f}%')

dc_type   shape                dtype           min        max       mean        std        p01        p99   %zero
main      (306, 83, 144, 8)    float64           0    3.4e+38    3.1e+34   3.25e+36       1.63   3.17e+03   0.00%


## Histograms

Subsamples projections (roughly 50 evenly spaced ones) so this stays fast on
large datasets.

In [17]:
@interact(dc_type=widgets.Dropdown(options=dc_types, description='dc_type'))
def _view_histograms(dc_type):
    dc = dcs[dc_type]
    stride = max(1, len(dc.projections) // 50)
    data = dc.data[::stride]
    has_diode = hasattr(dc, 'diode')

    fig, axes = plt.subplots(1, 2 if has_diode else 1, figsize=(10, 3.5))
    axes = np.atleast_1d(axes)

    vals = data.ravel()
    axes[0].hist(vals[vals > 0], bins=100, log=True, color='steelblue')
    axes[0].set_xlabel('SAXS intensity')
    axes[0].set_ylabel('count (log)')
    axes[0].set_title(f'{dc_type}: data distribution (every {stride}th proj.)')

    if has_diode:
        diode = dc.diode[::stride]
        axes[1].hist(diode.ravel(), bins=80, color='darkorange')
        axes[1].set_xlabel('diode (flux monitor)')
        axes[1].set_title(f'{dc_type}: diode distribution')

    plt.tight_layout()
    plt.show()

    s = projection_stats(dc, sample_stride=stride)
    print(f'range=[{s["min"]:.4g}, {s["max"]:.4g}]   mean={s["mean"]:.4g}   std={s["std"]:.4g}   '
          f'p01={s["p01"]:.4g}   p50={s["p50"]:.4g}   p99={s["p99"]:.4g}   '
          f'frac_zero={100 * s["frac_zero"]:.2f}%')

interactive(children=(Dropdown(description='dc_type', options=('main',), value='main'), Output()), _dom_classe…

## Per-frame intensity variation

How much does the per-frame max intensity vary across projections, and is a
large max-vs-percentile gap driven by a few outlier pixels rather than real
signal? Plots `max` alongside the 99.9th/99th/95th/50th percentile per frame
(log scale), plus the `max / p99` ratio — a spiky ratio points to outliers,
a flat-but-high one points to genuine per-frame signal variation.

In [18]:
_FRAME_PERCENTILES = (50, 95, 99, 99.9)


def per_frame_stats(dc, percentiles=_FRAME_PERCENTILES):
    data = dc.data.reshape(len(dc.projections), -1)
    stats = {'max': data.max(axis=1)}
    for p in percentiles:
        stats[f'p{p:g}'] = np.percentile(data, p, axis=1)
    return stats


@interact(dc_type=widgets.Dropdown(options=dc_types, description='dc_type'))
def _view_per_frame(dc_type):
    dc = dcs[dc_type]
    stats = per_frame_stats(dc)
    frames = np.arange(len(dc.projections))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(frames, stats['max'], label='max', color='crimson')
    for p in _FRAME_PERCENTILES[::-1]:
        axes[0].plot(frames, stats[f'p{p:g}'], label=f'p{p:g}')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('frame (projection index)')
    axes[0].set_ylabel('intensity (log)')
    axes[0].set_title(f'{dc_type}: per-frame intensity')
    axes[0].legend(fontsize=8)

    ratio = stats['max'] / stats['p99']
    axes[1].plot(frames, ratio, color='steelblue')
    axes[1].set_xlabel('frame (projection index)')
    axes[1].set_ylabel('max / p99')
    axes[1].set_title(f'{dc_type}: outlier ratio (max vs p99)')

    plt.tight_layout()
    plt.show()

    print(f'max : mean={stats["max"].mean():.4g}  std={stats["max"].std():.4g}  '
          f'min={stats["max"].min():.4g}  max={stats["max"].max():.4g}')
    print(f'p99 : mean={stats["p99"].mean():.4g}  std={stats["p99"].std():.4g}')
    print(f'max/p99 ratio : mean={ratio.mean():.4g}  std={ratio.std():.4g}  max={ratio.max():.4g}  '
          f'(large & spiky => a few outlier pixels; flat & high => genuine per-frame signal)')

interactive(children=(Dropdown(description='dc_type', options=('main',), value='main'), Output()), _dom_classe…